In [1]:
# Load NYC Yellow Taxi data from Lakehouse
df = spark.read.parquet("Files/sample_datasets/nyc_taxi_yellow_2020_1.parquet")

# Show first 10 rows
df.show(10)

# Show how many rows we have
print(f"Total rows: {df.count()}")

# Show column names
print(f"Columns: {df.columns}")

StatementMeta(, 6eb0cedd-90e5-47cd-83a8-44847b0b36f4, 3, Finished, Available, Finished, False)

+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+
|vendorID| tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount|
+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+
|       2|2020-01-03 03:57:12|2020-01-03 04:02:37|             1|        1.03|          97|          97|    NULL|    NULL|  NULL|  NULL|         1|              N|          1|       6.0|  1.0|   0.5|                 0.3|     1.95|     

In [2]:
# Load NYC Green Taxi data too
df_green = spark.read.parquet("Files/sample_datasets/nyc_taxi_green_2020_11.parquet")
print(f"Green taxi rows: {df_green.count()}")
print(f"Yellow taxi rows: {df.count()}")

StatementMeta(, 6eb0cedd-90e5-47cd-83a8-44847b0b36f4, 5, Finished, Available, Finished, False)

Green taxi rows: 2
Yellow taxi rows: 1


In [4]:
# Download NYC Taxi data using requests
import requests
import io

# Download the file
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
response = requests.get(url)

# Save to Lakehouse Files
with open("/lakehouse/default/Files/yellow_taxi_2023.parquet", "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")

# Now read it with Spark
df_yellow = spark.read.parquet("Files/yellow_taxi_2023.parquet")
print(f"Total trips: {df_yellow.count()}")
df_yellow.show(5)

StatementMeta(, 6eb0cedd-90e5-47cd-83a8-44847b0b36f4, 10, Finished, Available, Finished, False)

Downloaded successfully!
Total trips: 3066766
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|         141|  

In [5]:
# SILVER LAYER - Clean the data
from pyspark.sql.functions import col, hour, dayofweek, month

# Remove bad data
df_silver = df_yellow.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("total_amount") > 0)
)

# Add useful columns
df_silver = df_silver.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
                     .withColumn("pickup_day", dayofweek(col("tpep_pickup_datetime"))) \
                     .withColumn("pickup_month", month(col("tpep_pickup_datetime")))

print(f"Raw rows: {df_yellow.count()}")
print(f"Clean rows: {df_silver.count()}")
print(f"Removed: {df_yellow.count() - df_silver.count()} bad records")

StatementMeta(, 6eb0cedd-90e5-47cd-83a8-44847b0b36f4, 12, Finished, Available, Finished, False)

Raw rows: 3066766
Clean rows: 2884228
Removed: 182538 bad records


In [6]:
# Save Silver layer to Lakehouse
df_silver.write.mode("overwrite").saveAsTable("silver_taxi_trips")
print("Silver layer saved!")

# GOLD LAYER - Business insights
from pyspark.sql.functions import avg, sum, count, round

df_gold = df_silver.groupBy("pickup_hour").agg(
    count("*").alias("total_trips"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(sum("total_amount"), 2).alias("total_revenue")
).orderBy("pickup_hour")

df_gold.show(24)

# Save Gold layer
df_gold.write.mode("overwrite").saveAsTable("gold_hourly_stats")
print("Gold layer saved!")

StatementMeta(, 6eb0cedd-90e5-47cd-83a8-44847b0b36f4, 14, Finished, Available, Finished, False)

Silver layer saved!
+-----------+-----------+--------+------------+-------------+
|pickup_hour|total_trips|avg_fare|avg_distance|total_revenue|
+-----------+-----------+--------+------------+-------------+
|          0|      79567|    19.8|        4.03|   2297248.09|
|          1|      55565|   17.82|        3.49|    1461587.5|
|          2|      38731|   16.72|        3.22|    961081.34|
|          3|      25053|   17.74|        3.51|    651827.71|
|          4|      15836|   22.25|        4.68|    498478.22|
|          5|      16142|   26.46|        6.42|    591615.71|
|          6|      39945|   22.15|        4.78|   1221627.45|
|          7|      79552|   18.92|        3.68|   2133858.79|
|          8|     107901|   17.43|        3.18|   2710842.41|
|          9|     122644|   17.61|        3.09|   3129736.87|
|         10|     135285|   17.73|        3.14|   3463410.15|
|         11|     145262|   17.41|        3.04|   3658114.78|
|         12|     160233|   17.76|        3.11|   